In [2]:
!pip install --upgrade pip setuptools wheel
!pip -q install groq chromadb sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [1]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [2]:
import os
import uuid
from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import files
from pypdf import PdfReader

In [3]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Good small embedding model for Colab demos
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# In-memory Chroma client for demo
chroma_client = chromadb.EphemeralClient()

collection = chroma_client.create_collection(name="simple_rag_demo")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
uploaded = files.upload()

Saving 2026013520 (1).pdf to 2026013520 (1).pdf


In [6]:
def read_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_pdf(path):
    reader = PdfReader(path)
    text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)

docs = []

for fname in uploaded.keys():
    if fname.endswith(".txt"):
        docs.append({"source": fname, "text": read_txt(fname)})
    elif fname.endswith(".pdf"):
        docs.append({"source": fname, "text": read_pdf(fname)})
    else:
        print(f"Skipping unsupported file: {fname}")

print("Loaded documents:", len(docs))

Loaded documents: 1


In [7]:
def chunk_text(text, chunk_size=600, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []

for doc in docs:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "id": str(uuid.uuid4()),
            "text": chunk,
            "source": doc["source"],
            "chunk_id": i
        })

print("Total chunks:", len(all_chunks))

Total chunks: 49


In [8]:
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True).tolist()

collection.add(
    ids=[c["id"] for c in all_chunks],
    documents=chunk_texts,
    embeddings=chunk_embeddings,
    metadatas=[
        {"source": c["source"], "chunk_id": c["chunk_id"]}
        for c in all_chunks
    ]
)

print("Stored in vector DB.")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Stored in vector DB.


In [9]:
def retrieve_standard(query, top_k=4):
    query_embedding = embedder.encode([query]).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({
            "text": doc,
            "source": meta["source"],
            "chunk_id": meta["chunk_id"]
        })
    return retrieved

In [10]:
def answer_with_context(user_query, retrieved_docs, model="llama-3.1-8b-instant"):
    context = "\n\n".join(
        [f"[Source: {d['source']} | Chunk: {d['chunk_id']}]\n{d['text']}" for d in retrieved_docs]
    )

    prompt = f"""
You are a RAG assistant.
Answer only from the provided context.
If the answer is not in the context, say:
"Answer not found in the uploaded documents."

Context:
{context}

Question:
{user_query}
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Answer strictly from retrieved context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [12]:
query = input("Enter your question: ")

standard_docs = retrieve_standard(query, top_k=4)
standard_answer = answer_with_context(query, standard_docs)

print("\n=== STANDARD RAG: RETRIEVED CHUNKS ===\n")
for i, d in enumerate(standard_docs, 1):
    print(f"{i}. {d['source']} | chunk {d['chunk_id']}")
    print(d["text"][:250], "\n")

print("\n=== STANDARD RAG: ANSWER ===\n")
print(standard_answer)

Enter your question: what is forward forward algorithm

=== STANDARD RAG: RETRIEVED CHUNKS ===

1. 2026013520 (1).pdf | chunk 10
s follows: section II
discusses the previous work. section III describes the three
models we compare. Section IV shows our experimental setup,
and the results are in Section V followed by the conclusions
of the study in section VI.
II. BACKGROUND AND 

2. 2026013520 (1).pdf | chunk 0
Accelerating Forward-Forward Learning on Hybrid
Transfer Architectures via Adaptive Negative
Sampling
Shiva Dhanush S ∗, Sai Venkata Jaswant Kolupuri †, Satyatma Chincholi ‡, Shreyas Ghanathe §, and K C Narendra ¶
B.Tech Student∗†‡§ , Assistant Profe 

3. 2026013520 (1).pdf | chunk 11
le forward-backward pass of backpropagation with
two forward passes: one with positive (real) data and one
with negative (corrupted) data. Instead of one single global
loss, FF allows layers to train on a local goodness objective,
through which each  

4. 2026013520 (1).pdf | chunk 5
 namely the For

In [13]:
def generate_hypothetical_doc(query, model="llama-3.1-8b-instant"):
    prompt = f"""
Write a short, factual passage that would likely answer this question.
Do not mention that this is hypothetical.
Keep it concise and information-dense.

Question: {query}
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Generate a concise hypothetical reference passage."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

In [14]:
def retrieve_hyde(query, top_k=4):
    hypothetical_doc = generate_hypothetical_doc(query)
    hyp_embedding = embedder.encode([hypothetical_doc]).tolist()[0]

    results = collection.query(
        query_embeddings=[hyp_embedding],
        n_results=top_k
    )

    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({
            "text": doc,
            "source": meta["source"],
            "chunk_id": meta["chunk_id"]
        })

    return hypothetical_doc, retrieved

In [15]:
query = input("Enter your question for HyDE RAG: ")

hyde_doc, hyde_docs = retrieve_hyde(query, top_k=4)
hyde_answer = answer_with_context(query, hyde_docs)

print("\n=== HYPOTHETICAL DOCUMENT (HyDE) ===\n")
print(hyde_doc)

print("\n=== HyDE RAG: RETRIEVED CHUNKS ===\n")
for i, d in enumerate(hyde_docs, 1):
    print(f"{i}. {d['source']} | chunk {d['chunk_id']}")
    print(d["text"][:250], "\n")

print("\n=== HyDE RAG: ANSWER ===\n")
print(hyde_answer)

Enter your question for HyDE RAG: what is this pdf about

=== HYPOTHETICAL DOCUMENT (HyDE) ===

The PDF document, titled "Sustainable Energy Strategies for Urban Development," is a research report published by the International Energy Agency (IEA). It presents a comprehensive analysis of the current state of urban energy systems and provides recommendations for cities to transition towards more sustainable and resilient energy futures. The report focuses on the integration of renewable energy sources, energy efficiency measures, and smart grid technologies to support urban development and mitigate climate change impacts.

=== HyDE RAG: RETRIEVED CHUNKS ===

1. 2026013520 (1).pdf | chunk 19
ed
Forward-Forward (FF) framework in two main aspects. Firstly,
we carry out an efficient in-batch sampling to avoid redundant
data duplication. Secondly, we present an Adaptive Negative
Sampling (ANS) method that, by its false positive predictions,
 

2. 2026013520 (1).pdf | chunk 20
scard the final

In [16]:
def compare_methods(query):
    print("Running Standard RAG...\n")
    standard_docs = retrieve_standard(query, top_k=4)
    standard_answer = answer_with_context(query, standard_docs)

    print("Running HyDE RAG...\n")
    hyde_doc, hyde_docs = retrieve_hyde(query, top_k=4)
    hyde_answer = answer_with_context(query, hyde_docs)

    print("=" * 80)
    print("QUERY:")
    print(query)

    print("\n" + "=" * 80)
    print("STANDARD RAG ANSWER:\n")
    print(standard_answer)

    print("\n" + "=" * 80)
    print("HyDE GENERATED HYPOTHETICAL DOCUMENT:\n")
    print(hyde_doc)

    print("\n" + "=" * 80)
    print("HyDE RAG ANSWER:\n")
    print(hyde_answer)

# Example:
# compare_methods("What are the main objectives discussed in the uploaded document?")